# TELOS 3.1: DECLARE-TIMI 58 (NCT01730534) Pipeline

DECLARE-TIMI 58 임상시험 데이터를 사용한 TELOS 파이프라인 실행 노트북.

## Trial Overview
- **NCT ID**: NCT01730534
- **Title**: Multicenter Trial to Evaluate the Effect of Dapagliflozin on the Incidence of Cardiovascular Events (DECLARE-TIMI 58)
- **Target**: T2DM patients + CV risk factors → Dapagliflozin
- **Comparator**: T2DM patients + CV risk factors → Placebo
- **Primary Outcome**: MACE (Major Adverse Cardiovascular Events)

## Pipeline Overview

| Phase | Agent | Description |
|-------|-------|-------------|
| Cohort Definition | Agent 1 | Logic Decomposer (NCT → IR) |
| Cohort Definition | Agent 2 | Intelligent Mapper (IR → OMOP Concepts) |
| Cohort Definition | Agent 3 | Cohort Assembler (Concepts → Circe JSON) |
| Cohort Definition | Agent 4 | Validator (JSON Validation) |
| Evidence Generation | Agent 5 | Analysis Agent (Causal Inference) |
| Evidence Generation | Agent 6 | Reporting Agent (Visualization) |

---
## 0. Setup & Configuration

In [ ]:
# Add project root to path
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Configuration
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, HTML
import matplotlib.pyplot as plt

# Clinical Trial NCT ID (DECLARE-TIMI 58)
NCT_ID = "NCT01730534"

# Fetch trial data (auto-caches to data/nct_cache/)
from src.agents.agent1.nct_fetcher import fetch_or_load_trial_data
trial_data = fetch_or_load_trial_data(NCT_ID)

print(f"Trial: {trial_data.title}")
print("-" * 60)
print(f"Conditions: {', '.join(trial_data.conditions)}")
print(f"Interventions: {', '.join(trial_data.interventions)}")
print(f"Outcomes: {', '.join(trial_data.primary_outcomes)}")
print(f"Phase: {trial_data.phase}")
print(f"Inclusion: {len(trial_data.inclusion_criteria)} criteria")
print(f"Exclusion: {len(trial_data.exclusion_criteria)} criteria")

In [ ]:
# View raw eligibility criteria
print("📋 INCLUSION CRITERIA:")
for i, c in enumerate(trial_data.inclusion_criteria, 1):
    print(f"  {i}. {c[:120]}")

print("\n📋 EXCLUSION CRITERIA:")
for i, c in enumerate(trial_data.exclusion_criteria, 1):
    print(f"  {i}. {c[:120]}")

### Step 1: Agent 1 - Logic Decomposer

NCT eligibility criteria를 **Internal Representation (IR)** 구조로 변환합니다.

- **Input**: NCT ID (or cached JSON)
- **Output**: TELOSRequest (Target, Comparator, Outcome)

In [ ]:
# Agent 1: Logic Decomposer (NCT-based, with auto-caching)
try:
    from src.agents.agent1.parser import LogicDecomposer
    
    agent1 = LogicDecomposer()
    ir_result = agent1.parse_nct(NCT_ID)
    
    print("✅ Agent 1 Complete: IR Generated from NCT")
    print("="*60)
    
except Exception as e:
    print(f"⚠️ Agent 1 Error: {e}")
    import traceback
    traceback.print_exc()
    ir_result = None

In [ ]:
ir_result

In [ ]:
# View Agent 1 Results
if ir_result:
    print("📋 TARGET COHORT:")
    print(f"  - Primary Criteria: {ir_result.target.primary_criteria.entity_text}")
    print(f"  - Domain: {ir_result.target.primary_criteria.domain}")
    print(f"  - Limit: {ir_result.target.primary_criteria.limit}")
    print(f"  - Inclusion Rules: {len(ir_result.target.inclusion_rules)}")
    print(f"  - Exclusion Rules: {len(ir_result.target.exclusion_rules)}")
    
    print("\nInclusion Rules:")
    for r in ir_result.target.inclusion_rules:
        print(f"    - [{r.domain}] {r.entity_text} ({r.logic_type})")
    
    print("\nExclusion Rules:")
    for r in ir_result.target.exclusion_rules:
        print(f"    - [{r.domain}] {r.entity_text} ({r.logic_type})")

    print("\n📋 COMPARATOR COHORT:")
    print(f"  - Primary Criteria: {ir_result.comparator.primary_criteria.entity_text}")
    
    print("\n📋 OUTCOME:")
    print(f"  - Name: {ir_result.outcome.name}")
    print(f"  - Domain: {ir_result.outcome.domain}")
    print(f"  - Entity: {ir_result.outcome.entity_text}")
    print(f"  - Time-at-Risk: {ir_result.outcome.time_at_risk.start} to {ir_result.outcome.time_at_risk.end} days")

In [ ]:
# View Full IR as JSON
if ir_result:
    ir_json = ir_result.model_dump()
    print(json.dumps(ir_json, indent=2, default=str))

### Step 2: Agent 2 - Intelligent Mapper

IR의 텍스트 엔티티를 **OMOP Concept ID**로 매핑합니다.

- **Fast Path**: 단순 용어 → Direct Athena Lookup
- **Slow Path**: 복잡한 기준 → LLM Semantic Reranking

**Note**: ChromaDB와 DB 연결이 필요합니다.

In [ ]:
# Agent 2: Complexity Router Demo
from src.agents.agent2.complexity_router import ComplexityRouter

router = ComplexityRouter()

# Dynamically extract terms from IR
test_terms = []
if ir_result:
    # Primary criteria
    if ir_result.target.primary_criteria.entity_text:
        test_terms.append(ir_result.target.primary_criteria.entity_text)
    if ir_result.comparator.primary_criteria.entity_text:
        test_terms.append(ir_result.comparator.primary_criteria.entity_text)
    # Inclusion rules
    for rule in ir_result.target.inclusion_rules:
        if rule.entity_text:
            test_terms.append(rule.entity_text)
    # Exclusion rules
    for rule in ir_result.target.exclusion_rules:
        if rule.entity_text:
            test_terms.append(rule.entity_text)
    # Outcome
    if ir_result.outcome.entity_text:
        test_terms.append(ir_result.outcome.entity_text)

print("🔀 COMPLEXITY ROUTING RESULTS:")
print("="*60)
for term in test_terms:
    path = router.route(term)
    emoji = "🚀" if path == "fast" else "🐢"
    print(f"{emoji} [{path.upper():4}] {term}")


In [ ]:
# Agent 2: Full Mapping (requires ChromaDB)
try:
    from src.agents.agent2.workflow import Agent2Workflow
    
    agent2 = Agent2Workflow()
    
    # Extract ALL entities from IR
    if ir_result:
        entities_to_map = []
        
        # Primary criteria
        if ir_result.target.primary_criteria.entity_text:
            entities_to_map.append(ir_result.target.primary_criteria.entity_text)
        if ir_result.comparator.primary_criteria.entity_text:
            entities_to_map.append(ir_result.comparator.primary_criteria.entity_text)
        
        # Inclusion rules
        for rule in ir_result.target.inclusion_rules:
            if rule.entity_text:
                entities_to_map.append(rule.entity_text)
        
        # Exclusion rules
        for rule in ir_result.target.exclusion_rules:
            if rule.entity_text:
                entities_to_map.append(rule.entity_text)
        
        # Outcome
        if ir_result.outcome.entity_text:
            entities_to_map.append(ir_result.outcome.entity_text)
        
        entities_to_map = [e for e in entities_to_map if e]
        
        print(f"Mapping {len(entities_to_map)} entities from IR...")
        mapping_results = agent2.process_batch(entities_to_map)
        print(f"\n✅ Agent 2 Mapping Complete")
        print(f"  Mapped: {mapping_results.gap_report.mapped_count}/{len(entities_to_map)} entities")
        print(f"  Unique concept IDs: {len(mapping_results.concept_ids)}")
        print(f"  Fast path: {mapping_results.fast_path_count}, Slow path: {mapping_results.slow_path_count}")
    else:
        print("⚠️ No IR available - skipping mapping")
        
except Exception as e:
    print(f"⚠️ Agent 2 Error: {e}")
    import traceback
    traceback.print_exc()


### Step 3 & 4: Agent 3 (Assembler) & Agent 4 (Validator)

- **Agent 3**: ConceptSet + IR → Circe-be JSON
- **Agent 4**: JSON Schema Validation + Dry Run

In [ ]:
# Agent 3 & 4: Using Pipeline
try:
    from src.pipeline.cohort_pipeline import CohortPipeline
    
    cohort_pipeline = CohortPipeline()
    cohort_result = cohort_pipeline.run(NCT_ID)
    
    print("✅ Cohort Pipeline Complete")
    print(f"  Valid: {cohort_result.is_valid}")
    print(f"  ConceptSets: {len(cohort_result.circe_json.get('ConceptSets', []))}")
    
except Exception as e:
    print(f"⚠️ Cohort Pipeline Error: {e}")
    cohort_result = None

In [ ]:
# View Circe JSON Structure
if cohort_result and cohort_result.circe_json:
    circe = cohort_result.circe_json
    print("📄 CIRCE JSON STRUCTURE:")
    print(f"  - ConceptSets: {len(circe.get('ConceptSets', []))}")
    print(f"  - PrimaryCriteria: {bool(circe.get('PrimaryCriteria'))}")
    print(f"  - InclusionRules: {len(circe.get('InclusionRules', []))}")
    print(f"  - EndStrategy: {(circe.get('EndStrategy') or {}).get('DateOffset', 'N/A')}")
    
    # Save to file
    output_dir = PROJECT_ROOT / "output" / "declare_timi"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    with open(output_dir / "circe_cohort.json", "w") as f:
        json.dump(circe, f, indent=2)
    print(f"\n💾 Saved to: {output_dir / 'circe_cohort.json'}")

---
## Phase 2: Evidence Generation (Synthetic Data Demo)

실제 OMOP CDM 데이터 없이도 Agent 5-6을 테스트할 수 있도록
DECLARE-TIMI 58 시나리오에 맞는 합성 데이터를 생성합니다.

- **Target**: Dapagliflozin
- **Comparator**: Placebo
- **Outcome**: MACE (CV death, MI, Ischemic stroke)

### Step 5: Generate Synthetic Clinical Trial Data (DECLARE-TIMI 58)

In [ ]:
def generate_declare_timi_data(n_dapa: int = 8582, n_placebo: int = 8578, seed: int = 42):
    """Generate synthetic DECLARE-TIMI 58 clinical trial data.
    
    Based on published results:
    - HR for MACE: 0.93 (95% CI: 0.84-1.03, p=0.17) -> NOT significant
    - Median follow-up: 4.2 years
    """
    np.random.seed(seed)
    # Use smaller sample for demo (1/10 scale)
    n_dapa = n_dapa // 10
    n_placebo = n_placebo // 10
    n_total = n_dapa + n_placebo
    
    # Treatment assignment (Dapagliflozin vs Placebo)
    treatment = np.array([1] * n_dapa + [0] * n_placebo)
    
    # Covariates
    age_dapa = np.random.normal(64, 7, n_dapa)
    age_placebo = np.random.normal(64, 7, n_placebo)
    age = np.concatenate([age_dapa, age_placebo])
    
    # HbA1c (%) - baseline
    hba1c_dapa = np.random.normal(8.3, 1.2, n_dapa)
    hba1c_placebo = np.random.normal(8.3, 1.2, n_placebo)
    hba1c = np.concatenate([hba1c_dapa, hba1c_placebo])
    
    # BMI
    bmi = np.random.normal(32, 5, n_total)
    
    # eGFR (mL/min/1.73m²)
    egfr = np.random.normal(85, 20, n_total)
    egfr = np.clip(egfr, 30, 150)
    
    # Established CV disease (binary, ~40%)
    cv_disease = np.random.binomial(1, 0.4, n_total)
    
    # Simulate survival times
    # DECLARE-TIMI: HR ~ 0.93 for MACE, median follow-up 4.2 years (1533 days)
    baseline_hazard = 0.00015  # Low daily hazard for MACE
    treatment_hr = 0.93  # Dapagliflozin reduces MACE (non-significant)
    
    times = []
    events = []
    
    for i in range(n_total):
        risk_factor = 1 + 0.02 * (age[i] - 64) + 0.1 * cv_disease[i] + 0.01 * (hba1c[i] - 8.3)
        if treatment[i] == 1:  # Dapagliflozin
            risk_factor *= treatment_hr
        
        lambda_i = baseline_hazard * max(risk_factor, 0.1)
        t = np.random.exponential(1 / lambda_i)
        
        max_followup = 1533  # ~4.2 years in days
        if t > max_followup:
            times.append(max_followup)
            events.append(0)  # Censored
        else:
            times.append(t)
            events.append(1)  # MACE event
    
    return pd.DataFrame({
        "person_id": range(n_total),
        "treatment": treatment,
        "age": age,
        "hba1c": hba1c,
        "bmi": bmi,
        "egfr": egfr,
        "cv_disease": cv_disease,
        "time": times,
        "event": events
    })

# Generate data
patient_data = generate_declare_timi_data()

print("📊 SYNTHETIC DATA GENERATED (DECLARE-TIMI 58):")
print("="*60)
print(f"  Total patients: {len(patient_data)}")
print(f"  Dapagliflozin (treated): {(patient_data['treatment'] == 1).sum()}")
print(f"  Placebo (control): {(patient_data['treatment'] == 0).sum()}")
print(f"  MACE events: {patient_data['event'].sum()}")
print(f"  Event rate: {patient_data['event'].mean()*100:.1f}%")
print(f"  Established CV disease: {patient_data['cv_disease'].mean()*100:.1f}%")

In [ ]:
# View data summary
display(patient_data.describe().round(2))

In [ ]:
# Visualize covariate distributions by treatment group
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

covariates = ['age', 'hba1c', 'bmi', 'egfr']
titles = ['Age Distribution', 'HbA1c (%)', 'BMI (kg/m²)', 'eGFR (mL/min/1.73m²)']

for ax, cov, title in zip(axes.flat, covariates, titles):
    dapa = patient_data[patient_data['treatment'] == 1][cov]
    placebo = patient_data[patient_data['treatment'] == 0][cov]
    
    ax.hist(dapa, bins=20, alpha=0.6, label='Dapagliflozin', color='#2563eb')
    ax.hist(placebo, bins=20, alpha=0.6, label='Placebo', color='#dc2626')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.show()

### Step 6: Agent 5 - Analysis Agent (Causal Inference)

In [ ]:
# Agent 5: Analysis Workflow
from src.agents.agent5 import Agent5Workflow

agent5 = Agent5Workflow()
agent5.configure(
    target_cohort_id=1,      # Dapagliflozin
    comparator_cohort_id=2,  # Placebo
    analysis_method="IPTW"   # Inverse Probability of Treatment Weighting
)

print("🔬 Running Agent 5 (Causal Inference)...")
analysis_results = agent5.run(data=patient_data)

print("✅ Agent 5 Complete")

In [ ]:
# View Analysis Results
hr = analysis_results["hazard_ratio"]

print("📈 HAZARD RATIO ANALYSIS (DECLARE-TIMI 58):")
print("="*60)
print(f"  HR: {hr['hr']:.3f}")
print(f"  95% CI: [{hr['ci_lower']:.3f}, {hr['ci_upper']:.3f}]")
print(f"  p-value: {hr['p_value']:.4f}")
print()

# Interpretation
if hr['hr'] > 1:
    effect = "increased risk"
else:
    effect = "decreased risk"

if hr['p_value'] < 0.05:
    significance = "statistically significant"
else:
    significance = "NOT statistically significant"

print(f"📋 INTERPRETATION:")
print(f"  Dapagliflozin shows {effect} for MACE (HR = {hr['hr']:.2f})")
print(f"  Result is {significance} (p = {hr['p_value']:.4f})")
print(f"\n  Expected from trial: HR = 0.93 (95% CI: 0.84-1.03, p=0.17)")

In [ ]:
# View Covariate Balance
balance = analysis_results["balance"]

print("⚖️ COVARIATE BALANCE (SMD < 0.1 is acceptable):")
print("="*60)

balance_df = pd.DataFrame([
    {"Covariate": cov, "SMD Before": vals["smd_before"], "Balanced": "✅" if vals["balanced"] else "❌"}
    for cov, vals in balance.items()
])
display(balance_df)

### Step 7: Agent 6 - Reporting Agent (Visualization)

In [ ]:
# Agent 6: Reporting Workflow
from src.agents.agent6 import Agent6Workflow
from src.reporting.models import HazardRatioSummary

agent6 = Agent6Workflow()

hr_summary = HazardRatioSummary(
    hr=hr['hr'],
    ci_lower=hr['ci_lower'],
    ci_upper=hr['ci_upper'],
    p_value=hr['p_value']
)

agent6.set_results(
    study_title=f"TTE: {trial_data.title} ({NCT_ID})",
    hazard_ratio=hr_summary,
    target_n=analysis_results['n_target'],
    comparator_n=analysis_results['n_comparator'],
    balance=analysis_results['balance'],
    ps_scores=analysis_results['ps_scores'],
    treatment=analysis_results['treatment'],
    survival_data=analysis_results['survival_data']
)

print("✅ Agent 6 Configured")

In [ ]:
# Generate Plots
output_dir = PROJECT_ROOT / "output" / "declare_timi"
output_dir.mkdir(parents=True, exist_ok=True)

plot_paths = agent6.generate_plots(str(output_dir / "plots"))

print("📊 GENERATED PLOTS:")
for name, path in plot_paths.items():
    print(f"  - {name}: {path}")

In [ ]:
# Display Forest Plot
from IPython.display import Image

print("🌲 FOREST PLOT:")
display(Image(filename=plot_paths['forest']))

In [ ]:
# Display Kaplan-Meier Curve
print("📈 KAPLAN-MEIER SURVIVAL CURVES:")
display(Image(filename=plot_paths['km']))

In [ ]:
# Display Love Plot (Covariate Balance)
print("❤️ LOVE PLOT (Covariate Balance):")
display(Image(filename=plot_paths['love']))

In [ ]:
# Display Propensity Score Distribution
print("📊 PROPENSITY SCORE DISTRIBUTION:")
display(Image(filename=plot_paths['ps_dist']))

In [ ]:
# Generate HTML Report
report_path = str(output_dir / "report.html")
agent6.generate_html_report(report_path)

print(f"📄 HTML Report saved: {report_path}")
print(f"\n🔗 Open in browser: file://{report_path}")

---
## Summary

### Pipeline Execution Complete

In [ ]:
# Final Summary
print("="*70)
print("TELOS 3.1 PIPELINE: DECLARE-TIMI 58 EXECUTION SUMMARY")
print("="*70)
print(f"\n📋 Study: {trial_data.title}")
print(f"   NCT ID: {NCT_ID}")
print(f"   Conditions: {', '.join(trial_data.conditions)}")
print(f"   Interventions: {', '.join(trial_data.interventions)}")
print(f"\n📊 Sample Size:")
print(f"   - Target (Dapagliflozin): {analysis_results['n_target']}")
print(f"   - Comparator (Placebo): {analysis_results['n_comparator']}")
print(f"\n📈 Primary Outcome (MACE):")
print(f"   - Hazard Ratio: {hr['hr']:.2f} (95% CI: {hr['ci_lower']:.2f}-{hr['ci_upper']:.2f})")
print(f"   - p-value: {hr['p_value']:.4f}")
print(f"\n💡 Key Finding:")
if hr['p_value'] < 0.05:
    direction = "higher" if hr['hr'] > 1 else "lower"
    print(f"   Significantly {direction} MACE risk with Dapagliflozin vs Placebo.")
else:
    print(f"   No significant difference in MACE between Dapagliflozin and Placebo.")
    print(f"   (Consistent with actual DECLARE-TIMI 58 findings)")
print(f"\n📁 Outputs saved to: {output_dir}")
print("="*70)